```markdown
# Projet DoctAgri : Classification des maladies des plantes

Ce projet implémente un pipeline complet d'apprentissage par transfert (Transfer Learning) basé sur l'architecture de réseau de neurones **MobileNetV2** pré-entraîné sur le dataset ImageNet. L'objectif est de diagnostiquer automatiquement les maladies affectant différentes variétés de plantes à partir d'images de feuilles (en utilisant le dataset populaire PlantVillage).

Le pipeline est conçu pour s'exécuter efficacement sur un GPU (comme le T4 de Google Colab) et comprend :
1. **L'importation et l'extraction** du dataset depuis Google Drive.
2. **La préparation et la normalisation** des images via un générateur (`ImageDataGenerator`) gérant également la répartition (split) d'entraînement et de validation.
3. **L'adaptation de l'architecture** MobileNetV2 avec le gel du réseau de base, l'ajout de couches denses personnalisées et de Dropout pour prévenir le surapprentissage.
4. **La compilation et l'entraînement** optimisés avec Adam.
5. **L'évaluation** à travers la visualisation des courbes d'apprentissage.
6. **L'exportation finale** du modèle en format natif Keras et sa conversion optimisée en format léger **TensorFlow Lite (TFLite) quantifié en INT8** pour un déploiement efficace sur des appareils mobiles et embarqués.
```

```markdown
# Implémentation du modèle DoctAgri (Transfer Learning avec MobileNetV2)

Ce notebook contient le pipeline complet, propre et optimisé pour le GPU T4 : extraction des données, préparation des générateurs, définition de l'architecture, entraînement, évaluation et exportation finale (Keras et TFLite quantifié).
```

```markdown
## 1. Extraction du Dataset depuis Google Drive
```

In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/plantvilage.zip'
extract_path = '/content/plantvillage_doctagri'

os.makedirs(extract_path, exist_ok=True)

print("Extraction du dataset en cours...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print(f"Extraction terminée dans : {extract_path}")

Extraction du dataset en cours...
Extraction terminée dans : /content/plantvillage_doctagri


```markdown
## 2. Préparation des Générateurs de Données (Train & Validation)
```

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Détection automatique du répertoire contenant les classes de maladies
base_extract = '/content/plantvillage_doctagri'
subdirs = [os.path.join(base_extract, d) for d in os.listdir(base_extract) if os.path.isdir(os.path.join(base_extract, d))]

# Si le zip contenait un dossier racine intermédiaire, on descend d'un niveau
if len(subdirs) == 1:
    dataset_path = subdirs[0]
    # Cas fréquent où il y a un sous-dossier 'train' ou 'New Plant Diseases Dataset(Augmented)'
    second_level = [os.path.join(dataset_path, d) for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
    if any(d.endswith('train') or 'train' in d.lower() for d in os.listdir(dataset_path)):
        dataset_path = next(os.path.join(dataset_path, d) for d in os.listdir(dataset_path) if 'train' in d.lower())
    elif len(second_level) == 1:
        dataset_path = second_level[0]
else:
    dataset_path = base_extract

print(f"Le chemin final utilisé pour charger les classes est : {dataset_path}")

# Paramètres globaux
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Normalisation et split de 20% pour la validation intégrés dans le générateur
datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    validation_split=0.2
)

# Générateur d'entraînement
train_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

# Générateur de validation
val_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

# Récupération de la liste ordonnée des classes
class_names = sorted(list(train_generator.class_indices.keys()))
num_classes = len(class_names)
print(f"\nNombre de classes détectées : {num_classes}")
if num_classes > 0:
    print(f"Exemples de classes : {class_names[:5]}...")


```markdown
## 3. Architecture du Modèle (Transfer Learning MobileNetV2)
```

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout
from tensorflow.keras.applications import MobileNetV2

# Validation de la présence de classes valides
if num_classes <= 1:
    raise ValueError(f"Erreur : Seulement {num_classes} classe détectée. Veuillez vérifier le chemin d'extraction du dataset.")

# Chargement du backbone pré-entraîné sur ImageNet
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Geler l'intégralité des couches du backbone
base_model.trainable = False

# Construction du modèle séquentiel DoctAgri avec la bonne dimension de sortie (38 classes)
model = Sequential([
    Input(shape=(224, 224, 3)),
    base_model,
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 62720)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    32,113,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,371,649 (131.12 MB)

 Trainable params: 32,113,665 (122.50 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

```markdown
## 4. Compilation et Entraînement du Modèle
```

In [ ]:
from tensorflow.keras.optimizers import Adam

# Configuration de l'optimiseur selon le cahier des charges
optimizer = Adam(
    learning_rate=0.0001,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-07
)

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Entraînement
epochs = 10
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=val_generator,
    verbose=1
)

Epoch 1/10


/usr/local/lib/python3.13/dist-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/losses/losses.py:33: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(None, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return self.fn(y_true, y_pred, **self._fn_kwargs)


4073/4073 ━━━━━━━━━━━━━━━━━━━━ 11590s 3s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/10
 717/4073 ━━━━━━━━━━━━━━━━━━━━ 2:13:06 2s/step - accuracy: 1.0000 - loss: 0.0000e+00

```markdown
## 5. Visualisation des Performances (Courbes d'Accuracy et de Loss)
```

In [ ]:
import matplotlib.pyplot as plt

# Courbe d'accuracy
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Précision (Accuracy)')
plt.xlabel('Époque')
plt.ylabel('Score')
plt.legend()
plt.grid(True)

# Courbe de loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Perte (Loss)')
plt.xlabel('Époque')
plt.ylabel('Score')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

```markdown
## 6. Sauvegarde et Exportation Mobile (TFLite INT8 Dynamic Range)
```

In [ ]:
# Dossier de sauvegarde final dans Google Drive
output_dir = '/content/drive/MyDrive/plant_disease_models'
os.makedirs(output_dir, exist_ok=True)

keras_model_path = os.path.join(output_dir, 'plant_disease_model.keras')
tflite_model_path = os.path.join(output_dir, 'model.tflite')
labels_path = os.path.join(output_dir, 'labels.txt')

# 1. Sauvegarde du modèle Keras d'origine
model.save(keras_model_path)
print(f"Modèle Keras sauvegardé sous : {keras_model_path}")

# 2. Conversion au format TFLite avec quantification dynamique (INT8 Dynamic Range)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_quant_model = converter.convert()

with open(tflite_model_path, 'wb') as f:
    f.write(tflite_quant_model)
print(f"Modèle TFLite quantifié sauvegardé sous : {tflite_model_path}")

# 3. Export de la liste ordonnée des labels dans un fichier 'labels.txt'
with open(labels_path, 'w') as f:
    for label in class_names:
        f.write(f"{label}\n")
print(f"Fichier de labels généré avec succès sous : {labels_path}")